# Chess Elo Regression — Advanced Game, Clock + Engine Features V3

This notebook:
- Parses the **annotated Lichess PGN format** (`{ [%clk H:MM:SS] [%eval x.xx] }` comments)
- Extracts **47 hand-crafted features** via `chess_features_v3.py` including clock/time-management signals, newly added **Quality of Play features (ACPL, Blunder Counts)** and **Density Ratios**.
- Uses **inverse-frequency sample weights** so rare high-Elo games count more
- Uses **SimpleImputer** to naturally handle engine evaluation NaNs.
- Measures feature importance with **permutation importance** (parallelised with joblib)

In [1]:
# pip install chess zstandard lightgbm pandas scikit-learn matplotlib joblib

import io, re, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import zstandard as zstd
import lightgbm as lgb

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import OneHotEncoder
from scipy.stats import randint, loguniform
from joblib import Parallel, delayed

# IMPORT V3 FILE
from chess_features_v3 import extract_features_dataframe, compute_elo_sample_weights

print('All imports OK')

In [2]:
# ── global config ─────────────────────────────────────────────────────────────
DATA_PATH    = "../../data/villads_data/lichess_db_standard_rated_2017-11.pgn.zst"
MAX_GAMES    = 20_000
RANDOM_STATE = 42
TEST_SIZE    = 0.2
N_JOBS       = -1   # use all CPU cores

# Elo bin edges for stratified weighting
ELO_BINS = [0, 1000, 1200, 1400, 1600, 1800, 2000, 2200, 2500, 4000]

# Feature groups produced by chess_features_v3.py
NUMERIC_FEATURES = [
    "total_ply_count", "material_balance_end",
    "checks_given_white", "checks_given_black",
    "first_capture_move_white", "first_capture_move_black",
    "pawn_captures_total", "piece_captures_total",
    "castle_move_white", "castle_move_black",
    "result_encoded",
    "consec_same_piece_white", "consec_same_piece_black",
    "queen_moves_before_10",
    "white_territory_depth", "black_territory_depth",
    "promotions", "en_passant_captures",
    "legal_moves_white_move5", "legal_moves_black_move5",
    # Normalized clock features
    "avg_time_per_move_white_norm", "std_time_per_move_white_norm",
    "max_time_single_move_white_norm", "time_pressure_moves_white",
    "opening_pace_white_norm", "clock_remaining_white_norm",
    "avg_time_per_move_black_norm", "std_time_per_move_black_norm",
    "max_time_single_move_black_norm", "time_pressure_moves_black",
    "opening_pace_black_norm", "clock_remaining_black_norm",
    # New V3 Advanced Features
    "capture_density", "check_density_white", "check_density_black",
    "acpl_white", "inaccuracy_count_white", "mistake_count_white", 
    "blunder_count_white", "blunder_density_white",
    "acpl_black", "inaccuracy_count_black", "mistake_count_black", 
    "blunder_count_black", "blunder_density_black",
]
CATEGORICAL_FEATURES = ["castle_side_white", "castle_side_black"]
METADATA_FEATURES    = ["TimeControl", "ECO"]

LGBM_CONFIG = {
    "n_estimators":      500,
    "learning_rate":     0.05,
    "num_leaves":        63,
    "min_child_samples": 20,
    "subsample":         0.8,
    "colsample_bytree":  0.8,
    "random_state":      RANDOM_STATE,
    "verbose":           -1,
    "n_jobs":            N_JOBS,
}

## 1 · Parse annotated PGN
Read the full PGN block so `chess_features_v3.py` can extract clock times and engine ACPL tags.

In [3]:
games_list = []
dctx = zstd.ZstdDecompressor()

with open(DATA_PATH, "rb") as compressed_file:
    with dctx.stream_reader(compressed_file) as reader:
        text_stream = io.TextIOWrapper(reader, encoding="utf-8")
        current_game = {}

        for line in text_stream:
            line = line.strip()

            if line.startswith('['):
                tag = line.split(' ')[0][1:]
                val = line.split('"')[1]
                if tag in ['WhiteElo', 'BlackElo', 'TimeControl', 'ECO', 'Termination', 'Result']:
                    current_game[tag] = val

            elif line.startswith('1.'):
                current_game['Moves'] = line
                if 'WhiteElo' in current_game and 'BlackElo' in current_game:
                    games_list.append(current_game)
                current_game = {}
                if len(games_list) >= MAX_GAMES:
                    break

df_raw = pd.DataFrame(games_list)
df_raw['WhiteElo'] = pd.to_numeric(df_raw['WhiteElo'], errors='coerce')
df_raw['BlackElo'] = pd.to_numeric(df_raw['BlackElo'], errors='coerce')
df_raw = df_raw.dropna(subset=['WhiteElo', 'BlackElo']).copy()
df_raw['WhiteElo'] = df_raw['WhiteElo'].astype(int)
df_raw['BlackElo'] = df_raw['BlackElo'].astype(int)
df_raw['Moves']    = df_raw['Moves'].fillna('').astype(str)

print(f"Loaded {len(df_raw):,} games")

In [4]:
_CLK_RE = re.compile(r'\[%clk\s+(\d+):(\d+):(\d+)\]')

def parse_clock_features(moves_string: str, time_control: str = '?') -> dict:
    clocks = [
        int(h)*3600 + int(m)*60 + int(s)
        for h, m, s in _CLK_RE.findall(moves_string)
    ]

    clk_w = clocks[0::2]  
    clk_b = clocks[1::2]  

    tc_m = re.match(r'(\d+)\+(\d+)', str(time_control))
    base      = int(tc_m.group(1)) if tc_m else None
    increment = int(tc_m.group(2)) if tc_m else 0

    def time_spent(seq, start):
        if not seq:
            return []
        spent = []
        prev = start
        for clk in seq:
            if prev is not None:
                s = prev - clk + increment
                if s >= 0:
                    spent.append(s)
            prev = clk
        return spent

    spent_w = time_spent(clk_w, base)
    spent_b = time_spent(clk_b, base)

    def stats(seq):
        if not seq:
            return 0.0, 0.0, 0.0, 0.0
        a = np.array(seq)
        return float(a.mean()), float(a.std()), float(a.max()), float(a[-1])

    aw, sw, mw, _ = stats(spent_w)
    ab, sb, mb, _ = stats(spent_b)

    return {
        'avg_time_per_move_white':    aw,
        'std_time_per_move_white':    sw,
        'max_time_single_move_white': mw,
        'time_pressure_moves_white':  sum(1 for c in clk_w if c < 10),
        'opening_pace_white':         float(np.mean(spent_w[:10])) if spent_w else 0.0,
        'clock_remaining_white':      clk_w[-1] if clk_w else 0.0,
        'avg_time_per_move_black':    ab,
        'std_time_per_move_black':    sb,
        'max_time_single_move_black': mb,
        'time_pressure_moves_black':  sum(1 for c in clk_b if c < 10),
        'opening_pace_black':         float(np.mean(spent_b[:10])) if spent_b else 0.0,
        'clock_remaining_black':      clk_b[-1] if clk_b else 0.0,
    }

tc_col = df_raw['TimeControl'] if 'TimeControl' in df_raw.columns else ['?'] * len(df_raw)
clock_records = Parallel(n_jobs=N_JOBS)(
    delayed(parse_clock_features)(moves, tc)
    for moves, tc in zip(df_raw['Moves'], tc_col)
)
df_clocks = pd.DataFrame(clock_records, index=df_raw.index)

## 2 · Extract features (parallel)

In [5]:
t0       = time.time()
df_feats = extract_features_dataframe(df_raw, n_jobs=N_JOBS)
elapsed  = time.time() - t0
print(f"Extracted {len(df_feats):,} games in {elapsed:.1f}s  "
      f"({elapsed/len(df_feats)*1000:.1f} ms/game)")

In [6]:
# Merge targets + metadata with extracted features and regex clock features
clock_cols = [
    "avg_time_per_move_white", "std_time_per_move_white", "max_time_single_move_white",
    "time_pressure_moves_white", "opening_pace_white", "clock_remaining_white",
    "avg_time_per_move_black", "std_time_per_move_black", "max_time_single_move_black",
    "time_pressure_moves_black", "opening_pace_black", "clock_remaining_black",
]

cols_to_drop = [col for col in clock_cols if col in df_feats.columns]
df_feats_no_clock = df_feats.drop(columns=cols_to_drop)

df = (
    df_raw[["WhiteElo", "BlackElo"] + METADATA_FEATURES]
    .join(df_feats_no_clock, how="inner")
    .join(df_clocks, how="inner")
)

base_seconds = (
    df["TimeControl"].astype(str).str.extract(r"(\d+)\+")[0].astype(float)
)

df["avg_time_per_move_white_norm"] = df["avg_time_per_move_white"] / base_seconds
df["std_time_per_move_white_norm"] = df["std_time_per_move_white"] / base_seconds
df["max_time_single_move_white_norm"] = df["max_time_single_move_white"] / base_seconds
df["opening_pace_white_norm"] = df["opening_pace_white"] / base_seconds
df["clock_remaining_white_norm"] = df["clock_remaining_white"] / base_seconds

df["avg_time_per_move_black_norm"] = df["avg_time_per_move_black"] / base_seconds
df["std_time_per_move_black_norm"] = df["std_time_per_move_black"] / base_seconds
df["max_time_single_move_black_norm"] = df["max_time_single_move_black"] / base_seconds
df["opening_pace_black_norm"] = df["opening_pace_black"] / base_seconds
df["clock_remaining_black_norm"] = df["clock_remaining_black"] / base_seconds

df = df.replace([np.inf, -np.inf], np.nan)

## 3 · Balanced sample weights

In [7]:
weights_white = compute_elo_sample_weights(df["WhiteElo"], ELO_BINS)
weights_black = compute_elo_sample_weights(df["BlackElo"], ELO_BINS)

## 4 · Build feature matrix

In [8]:
def build_matrix(frame: pd.DataFrame):
    """Build a feature matrix with robust median imputation + one-hot encoding."""
    numeric_features = NUMERIC_FEATURES
    categorical_features = CATEGORICAL_FEATURES + METADATA_FEATURES

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ],
        remainder="drop",
    )

    X_local = preprocessor.fit_transform(frame)
    feature_names = list(preprocessor.get_feature_names_out())
    return X_local, feature_names


X, all_feature_names = build_matrix(df)
print(f"Feature matrix: {X.shape}  ({len(all_feature_names)} named features)")

## 5 · Train with balanced weights

In [9]:
def run_experiment(target_col: str, sample_weights: np.ndarray) -> dict:
    y = df[target_col].values

    idx = np.arange(len(y))
    idx_train, idx_test = train_test_split(
        idx, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    X_train, X_test = X[idx_train], X[idx_test]
    y_train, y_test = y[idx_train], y[idx_test]
    w_train         = sample_weights[idx_train]

    model = lgb.LGBMRegressor(**LGBM_CONFIG)
    model.fit(X_train, y_train, sample_weight=w_train)
    preds = model.predict(X_test)

    mae  = mean_absolute_error(y_test, preds)
    rmse = root_mean_squared_error(y_test, preds)
    print(f"{target_col}  →  MAE: {mae:.1f}   RMSE: {rmse:.1f}")

    return dict(model=model, X_train=X_train, X_test=X_test,
                y_train=y_train, y_test=y_test,
                w_train=w_train, preds=preds,
                mae=mae, rmse=rmse, label=target_col,
                idx_test=idx_test)

white_run = run_experiment("WhiteElo", weights_white)
black_run = run_experiment("BlackElo", weights_black)

## 6 · Permutation importance

In [10]:
def compute_perm_importance(run: dict, n_repeats: int = 5) -> pd.DataFrame:
    print(f"Computing permutation importance for {run['label']} "
          f"({n_repeats} repeats, {N_JOBS} workers)...")
    t0 = time.time()

    X_eval = run["X_test"].toarray() if hasattr(run["X_test"], "toarray") else run["X_test"]

    result = permutation_importance(
        run["model"],
        X_eval,
        run["y_test"],
        n_repeats=n_repeats,
        scoring="neg_mean_absolute_error",
        n_jobs=N_JOBS,
        random_state=RANDOM_STATE,
    )

    n = len(all_feature_names)
    df_imp = pd.DataFrame({
        "feature":    all_feature_names[:n],
        "importance": result.importances_mean[:n],
        "std":        result.importances_std[:n],
    }).sort_values("importance", ascending=False).reset_index(drop=True)
    return df_imp

perm_white = compute_perm_importance(white_run, n_repeats=5)
perm_black = compute_perm_importance(black_run,  n_repeats=5)